# Aula 10 · Projeto prático: visão geral

## O que é
### O cenário
A **GrowDev Crédito** é uma fintech que oferece contas e empréstimos. Hoje, um analista roda os notebooks na mão toda manhã para atualizar o relatório de inadimplência. Às vezes esquece, às vezes roda fora de ordem.

**Nossa missão:** entregar um **job** que atualiza os dados sozinho, valida a qualidade, decide o que publicar e avisa o time quando algo dá errado.

### As fontes de dados
Arquivos Parquet que chegam no volume `growdev.bronze.landing`:

| Arquivo | Conteúdo | Linhas |
|---|---|---|
| `customers` | Clientes (nome, estado, segmento, renda) | ~10 mil |
| `accounts` | Contas dos clientes | ~25 mil |
| `account_balances` | Saldo diário de cada conta | ~750 mil |
| `loans` | Empréstimos (valor, categoria, status, inadimplência) | ~36 mil |

> Os dados chegam **sujos de propósito**: datas em vários formatos, valores como `R$ 1.234,56` e `N/A`, estados por extenso ou por sigla.

### Os notebooks (pasta `Projeto/`)

| Notebook | Camada | Papel |
|---|---|---|
| `00_preparar_ambiente` | - | Cria catálogo, schemas, volume e copia os dados |
| `01_bronze` | Bronze | Ingere só os arquivos novos |
| `02_silver` | Silver | Limpa e faz upsert (`MERGE`) |
| `03_gold` | Gold | Indicadores por cliente e taxa de inadimplência |
| `04_validar_qualidade` | Silver | Mede a qualidade e publica um task value |
| `05_registrar_alerta_qualidade` | Gold | Registra quando a qualidade reprova |
| `06_alerta_risco` | Gold | Registra quando a inadimplência passa do limite |
| `07_registrar_execucao` | Gold | Log de toda execução |
| `utilitarios/simular_chegada_lote` | - | Entrega um arquivo novo na landing |

## Por que importa
Aprender construindo um caso próximo do real fixa o conteúdo e gera uma **peça de portfólio**.

## Como aplicar
### 1. Preparar o ambiente
Se você ainda não fez na aula 01, rode `Projeto/00_preparar_ambiente`.

### 2. Rodar o ponto de partida manualmente

In [ ]:
for notebook in ["00_preparar_ambiente", "01_bronze", "02_silver", "03_gold"]:
    dbutils.notebook.run(f"../Projeto/{notebook}", 900, {"catalogo": "growdev"})
    print(f"OK: {notebook}")

In [ ]:
%sql
SELECT * FROM growdev.gold.resumo_clientes ORDER BY valor_total_emprestado DESC LIMIT 10

### 3. O roteiro: o que cada agrupamento adiciona ao job

```
Agrupamento 3   preparar_ambiente → bronze → silver → gold               (o job base)
Agrupamento 4   + agendamento e gatilho por chegada de arquivo           (roda sozinho)
Agrupamento 5   + For each na bronze, validação, If/else e Run if        (decide e repete)
Agrupamento 6   + retentativas, timeout, alertas e Repair run            (pronto para produção)
Agrupamento 7   + jobs modulares e ingestão com Lakeflow Connect         (bônus)
```

O job final, ao fim do Agrupamento 6:

```
preparar_ambiente → bronze (For each: 4 tabelas) → silver → validar_qualidade → qualidade_ok?
                                                                    ├─ sim → gold → risco_alto? ─ sim → alerta_risco
                                                                    └─ não → registrar_alerta_qualidade
                                                  (sempre, Run if: All done) → registrar_execucao
```

## Checklist da aula
- [ ] Conheço o cenário de negócio e as fontes de dados
- [ ] Rodei os notebooks do projeto manualmente
- [ ] Sei qual job vou entregar e o que cada agrupamento adiciona